# Simulator vs real A100, across traffic conditions

Each traffic condition is run on the real GPU and in the simulator, with **identical requests** (same generator, same seed):

| line in the plots | what it is |
|---|---|
| **real A100** (black) | our scheduler drives real GPT-2 on Colab's A100, one forward pass per request per step, while NVML records power |
| **simulator** (yellow) | the normal simulator: `run_engine` with its own scheduler and clock |

Under each power plot is its **traffic**: the requests arriving, and the forward passes run in each window, prefill in purple and decode in green.

The summary table also keeps a third view, the simulator priced on the exact steps the real GPU ran (`*_same_steps` columns). It tests the power model on its own and is no longer plotted.

There are three mixed-traffic conditions, each plotted in 1 s windows.

| condition | traffic | what it shows |
|---|---|---|
| `light` | 40 mixed requests, about 2 per second | requests rarely overlap: busy, then idle |
| `bursty` | 60 mixed requests in about 0.4 s (the simulator notebook's traffic) | one long decode backlog |
| `spiky` | 6 walls of 12 long prompts, 4 s apart | prefill spikes, then quiet decode |


**What this version adds to the simulator** (each switchable in section 0):
- `EXTRAS = True`: the kernels the 242-entry template leaves out, including the LM head: 338 kernels per prefill pass and 326 per decode step, as profiled.
- `MEASURE_PRICES = True`: the LM head and the two mask kernels are timed on this card (section 6b) and priced from that, because EnergAIzer is 1.6-3.8x off on them.
- `HOST_TIME = True`: each simulated step also lasts at least the CPU time a real step takes (fitted in section 5b), so the simulator keeps real time under load.

**Outputs**, downloaded as `sim_vs_real.zip`: for every condition, `real_CONDITION_iterations.csv` and `sim_CONDITION_iterations.csv` with the **same 34 columns** as `Bucketing_measured_iterations.csv`, plus `sim_same_steps_CONDITION_iterations.csv`, a 1 s power series, a summary table, the measured prices and the figures.

**Reading the real CSV.** Batch, token, KV and FLOP columns are exact. Time columns are measured. `energy_j` and `avg_power_w` come from NVML, which on an A100 only updates about every 100 ms, so per-step power is smoothed at that scale: trust windows of 0.5 s or more. `peak_power_w` is the highest NVML reading during the step (the simulator's is its hottest kernel, a different thing). The three energy-split columns are blank on the real side, because a power meter cannot split energy by kernel type.

**Needs an A100 runtime** (Runtime → Change runtime type → A100). Roughly 40-50 minutes with all nine conditions: each real run takes under a minute, building EnergAIzer about 3 minutes, the price measurements about 3 minutes, and pricing new kernel shapes the rest.

## 0 · Settings

Only the three mixed-traffic conditions run, at their normal length with 1 s windows, as in the Qwen3 notebook. About 15-20 minutes.

`EXTRAS = True` adds the kernels the 242-entry template leaves out (`dynshape.extras`, on the GitHub branch `extra-kernels`). `MEASURE_PRICES = True` times the LM head and the two mask kernels on this card (section 6b) and prices them from that.

In [ ]:
CONDITIONS = {
    # Mixed traffic: prefill and decode together.
    'light':  dict(kind='gamma', n=40, qps=2.0, cv=1.0),      # requests rarely overlap
    'bursty': dict(kind='gamma', n=60, qps=100.0, cv=2.0),    # all arrive at once: a decode backlog
    'spiky':  dict(kind='bursts', bursts=6, per_burst=12, spacing_s=4.0,
                   prompt=896, decode=16),                     # walls of long prompts, 4 s apart
}
BASE = ['light', 'bursty', 'spiky']
RUN        = BASE
REPEATS    = 1      # real runs per condition; each gets its own CSV
BUCKET     = 32     # simulator: decode contexts rounded to multiples of this (1 = exact, slow)
EXTRAS     = True   # simulator: also price the kernels the template leaves out (dynshape.extras)
MEASURE_PRICES = True   # time the LM head and the mask kernels on this card, price them from that
HOST_TIME  = True   # simulator: charge the CPU time per step, fitted from the real runs (5b)
HOST_WAIT_W = 0.5   # GPU power above idle while it waits for the CPU (measured: +0.49 W)
BRANCH     = 'extra-kernels'   # GitHub branch to clone ('main' once extras are merged)
TARGET_MHZ = 900    # the clock EnergAIzer's tables were measured at
PLOT_DT_S  = 1.0    # default window; each scaled condition sets its own (dt)
MAX_RUN_S  = 600    # safety stop for one real run

## 1 · Setup

In [ ]:
import os, subprocess, sys, time, json, contextlib, dataclasses, collections

REPO = 'https://github.com/shubhamOjha1000/dynamic_shape_power_sim.git'
DIR  = '/content/dynamic_shape_power_sim'
if not os.path.isdir(DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, DIR], check=True)
else:
    subprocess.run(['git', '-C', DIR, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', DIR, 'checkout', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
if DIR not in sys.path:
    sys.path.insert(0, DIR)
os.chdir(DIR)
head = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print(f'{BRANCH}: {head}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nvidia-ml-py', 'transformers>=4.44,<5', 'pytest',
                'pyyaml', 'scipy', 'scikit-learn', 'cvxpy', 'opt-einsum', 'tqdm', 'gdown'],
               check=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, dynshape

XS = None                      # the extra-kernel list the simulator prices, or None
if EXTRAS:
    try:
        from dynshape.extras import gpt2_hf_eager
    except ImportError as exc:
        raise ImportError(f'EXTRAS needs dynshape.extras -- is the branch {BRANCH} pushed?') from exc
    XS = gpt2_hf_eager()

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

extras_msg = XS.name if XS is not None else 'off'
print('dynshape', dynshape.__version__, '| torch', torch.__version__, '| extra kernels:', extras_msg)
assert torch.cuda.is_available(), 'no GPU -- Runtime > Change runtime type > A100'
print('device  ', torch.cuda.get_device_name(0))

In [ ]:
# The extra-kernel code's own unit tests (CPU, under a minute): a quick check before the long run.
if EXTRAS:
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--tb=short', '-rf',
                        '-p', 'no:cacheprovider', 'tests/test_extras.py'],
                       capture_output=True, text=True)
    for line in (r.stdout + r.stderr).strip().splitlines()[-30:]:
        print(' ', line)
    verdict = 'all passed' if r.returncode == 0 else 'SOME FAILED -- the rest still runs; send this output'
    print(f'unit tests for the extras: {verdict}   [{time.time() - t0:.0f} s]')

## 2 · Lock the clock, and measure this card's idle floor

EnergAIzer's tables were measured at 900 MHz, so the card is locked there; persistence mode keeps the lock while nothing runs. The idle floor is measured rather than assumed, because Colab's card is an A100-**SXM4** and the tables come from an A100-**PCIe**.

The sampler records board power, NVML's cumulative energy counter and the SM clock every 10 ms.

In [ ]:
import threading, pynvml

def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode == 0, (r.stdout + r.stderr).strip()

for cmd in (['nvidia-smi', '-pm', '1'], ['nvidia-smi', '-lgc', f'{TARGET_MHZ},{TARGET_MHZ}']):
    ok, msg = sh(cmd)
    print(' '.join(cmd), '->', 'ok' if ok else msg[:150])

pynvml.nvmlInit()
HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
CARD = pynvml.nvmlDeviceGetName(HANDLE)
CARD = CARD.decode() if isinstance(CARD, bytes) else CARD

def _energy_j():
    return pynvml.nvmlDeviceGetTotalEnergyConsumption(HANDLE) / 1000.0     # mJ -> J

try:
    _energy_j()
    HAVE_ENERGY = True
except pynvml.NVMLError:
    HAVE_ENERGY = False


class NvmlSampler:
    '''Board power, the cumulative energy counter and the SM clock, every interval_s.'''

    def __init__(self, interval_s=0.01):
        self.interval_s = interval_s
        self._stop = threading.Event()

    def _loop(self):
        while not self._stop.is_set():
            try:
                t = time.perf_counter() - self.t0
                p = pynvml.nvmlDeviceGetPowerUsage(HANDLE) / 1000.0
                e = _energy_j() if HAVE_ENERGY else np.nan
                c = pynvml.nvmlDeviceGetClockInfo(HANDLE, pynvml.NVML_CLOCK_SM)
                self.rows.append((t, p, e, c))
            except pynvml.NVMLError:
                self.errors += 1
            self._stop.wait(self.interval_s)

    def start(self):
        self.rows, self.errors = [], 0
        self._stop.clear()
        self.t0 = time.perf_counter()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self):
        self._stop.set()
        self._thread.join(2.0)
        df = pd.DataFrame(self.rows, columns=['t_s', 'power_w', 'energy_j', 'sm_mhz'])
        df['energy_j'] -= df['energy_j'].iloc[0]
        return df


time.sleep(2)
counter_msg = 'yes' if HAVE_ENERGY else 'NOT SUPPORTED -- energy comes from integrating power'
print()
print(f'card           : {CARD}')
print(f'energy counter : {counter_msg}')

idle_df = NvmlSampler(0.02).start()
time.sleep(30)
idle_df = idle_df.stop()
IDLE_W = float(idle_df.power_w.median())
print(f'idle at {TARGET_MHZ} MHz : {IDLE_W:.2f} W median over 30 s, '
      f'clock {idle_df.sm_mhz.median():.0f} MHz')
print(f'EnergAIzer tables (A100-PCIe, 900 MHz): 47.35 W, difference {IDLE_W - 47.35:+.2f} W')

# Does the lock hold under load? A matmul loop for 3 s.
x = torch.randn(4096, 4096, device='cuda', dtype=torch.bfloat16)
load_df = NvmlSampler(0.02).start()
t_end = time.time() + 3
while time.time() < t_end:
    for _ in range(20):
        x @ x
    torch.cuda.synchronize()
load_df = load_df.stop()
del x
LOAD_MHZ = float(load_df.sm_mhz.median())
print(f'under load     : {LOAD_MHZ:.0f} MHz median, {load_df.power_w.median():.0f} W')
if abs(LOAD_MHZ - TARGET_MHZ) > 15:
    print('WARNING: the clock did not hold under load -- compare shapes, not absolute watts.')

## 3 · The traffic

Every run, real or simulated, regenerates its requests from the same seed, so both sides see identical requests. The scheduler settings are shared by both sides too.

- **Decode-heavy set:** total length is capped at 960 tokens, because real GPT-2 cannot go past 1,024. `spiky` uses a token budget big enough for a whole wall of prompts to land in one step.
- **Prefill-only set:** every prompt is exactly 1,000 tokens, and the token budget per step is a whole number of prompts (4 for `prefill_light` and `prefill_bursty`, a whole wall of 24 for `prefill_spiky`). So no prompt is ever split into small chunks, and every forward pass is a full 1,000-token prefill.

In [ ]:
from dynshape import (TrafficConfig, generate_traffic, TraceInterval, FixedLength,
                      build_replay_traffic, traffic_summary, check_fits_context,
                      GPT2_MAX_POSITIONS, SchedulerConfig)

def make_traffic(name, seed=0):
    '''Fresh request objects for one condition. The scheduler mutates them, so every
    run, real or simulated, gets its own copy from the same seed.'''
    c = CONDITIONS[name]
    if 'prompt' not in c:                     # zipf lengths, as in the simulator notebook
        return build_replay_traffic(num_requests=c['n'], qps=c['qps'], cv=c['cv'],
                                    min_tokens=64, max_total_tokens=960, theta=0.85,
                                    prefill_to_decode_ratio=4.0, seed=seed)
    lengths = FixedLength(c['prompt'], c['decode'])
    if c['kind'] == 'gamma':
        cfg = TrafficConfig(interval='gamma', qps=c['qps'], cv=c['cv'],
                            length_generator=lengths, num_requests=c['n'], seed=seed)
    else:
        rng = np.random.default_rng(seed)
        times = sorted(t for b in range(c['bursts'])
                       for t in b * c['spacing_s'] + rng.uniform(0, 0.005, c['per_burst']))
        cfg = TrafficConfig(interval_generator=TraceInterval(times), length_generator=lengths,
                            num_requests=len(times), seed=seed)
    reqs = generate_traffic(cfg)
    check_fits_context(reqs, GPT2_MAX_POSITIONS)
    return reqs

def sched_for(name):
    '''One scheduler config per condition, shared by BOTH sides. With fixed-length
    prompts the token budget is a whole number of prompts, so a prompt is never split
    into small chunks -- every prefill kernel is full size.'''
    c = CONDITIONS[name]
    if 'prompt' not in c:
        chunk = 2048                                          # the simulator notebook's setting
    elif c['kind'] == 'bursts':
        chunk = c['per_burst'] * c['prompt'] + (512 if c['decode'] > 1 else 0)   # a whole wall per step
    else:
        chunk = 4 * c['prompt'] if c['decode'] <= 1 else max(4 * c['prompt'], 512)
        # up to 4 whole prompts per step; decode-only traffic needs room for its decodes
    return SchedulerConfig(chunk_size=chunk, max_num_seqs=256, block_size=16, max_tokens=1024)

TRAFFIC = pd.DataFrame({name: traffic_summary(make_traffic(name)) for name in RUN})
display(TRAFFIC.round(2))
for name in RUN:
    print(f'{name:15s} token budget per step: {sched_for(name).chunk_size:,}')

## 4 · Real GPT-2, and the kernels it launches

GPT-2 124M in bf16 with **eager** attention, because the simulator's templates were traced from eager attention. The executor runs **one forward pass per request per step**, each request with its own KV cache, which is exactly what the simulator prices with `fuse_linear=False`. Outputs are meaningless (random prompt tokens, greedy decoding): this is a power benchmark.

The second cell counts real CUDA kernels per forward pass against the template's 242 entries and **lists them by name**. The last run found 326-350, and this list shows what the extra ones are.

In [ ]:
from transformers import GPT2LMHeadModel, DynamicCache
from dynshape import prompt_token_ids

DEV = 'cuda'
model = GPT2LMHeadModel.from_pretrained('gpt2', torch_dtype=torch.bfloat16,
                                        attn_implementation='eager').to(DEV).eval()
torch.set_grad_enabled(False)
print('attention:', model.config._attn_implementation, '| window:', model.config.n_positions)

for _ in range(3):              # cuBLAS handles, autotuning, allocator growth
    model(input_ids=torch.zeros(1, 128, dtype=torch.long, device=DEV))
torch.cuda.synchronize()


def gpu_sync():
    torch.cuda.synchronize()


class Executor:
    '''One real forward pass per Piece, each request holding its own KV cache.'''

    def __init__(self):
        self.cache, self.prompt, self.next_token = {}, {}, {}

    def admit(self, rid, n_prompt):
        self.prompt[rid] = torch.tensor(prompt_token_ids(n_prompt, seed=rid),
                                        dtype=torch.long, device=DEV)

    def release(self, rid):
        for d in (self.cache, self.prompt, self.next_token):
            d.pop(rid, None)

    def run_piece(self, piece):
        rid = piece.request_id
        if piece.kind == 'prefill':
            if piece.context == 0:                  # fresh start, or a restart after preemption
                self.cache[rid] = DynamicCache()
            ids = self.prompt[rid][piece.context: piece.context + piece.tokens][None, :]
        else:
            ids = self.next_token[rid].view(1, 1)
        out = model(input_ids=ids, past_key_values=self.cache[rid], use_cache=True)
        self.cache[rid] = out.past_key_values
        self.next_token[rid] = out.logits[0, -1].argmax()

print('model and executor ready')

In [ ]:
from torch.profiler import profile, ProfilerActivity

ACTIVE = 3

def profile_pass(fn, warmup=3):
    '''(CUDA kernels per pass, GPU ms per pass, kernel-name counts over ACTIVE passes).'''
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for _ in range(ACTIVE):
            fn()
        torch.cuda.synchronize()
    prof.export_chrome_trace('/content/kernel_trace.json')
    ev = json.load(open('/content/kernel_trace.json'))['traceEvents']
    ker = [e for e in ev if e.get('cat') in ('kernel', 'gpu_memcpy', 'gpu_memset')]
    names = collections.Counter(e['name'] for e in ker)
    return len(ker) / ACTIVE, sum(e['dur'] for e in ker) / ACTIVE / 1000.0, names

def one_prefill(n):
    ids = torch.zeros(1, n, dtype=torch.long, device=DEV)
    return lambda: model(input_ids=ids, use_cache=True)

def one_decode(ctx):
    # The cache grows by one token per call -- irrelevant for counting kernels.
    cache = model(input_ids=torch.zeros(1, ctx, dtype=torch.long, device=DEV),
                  use_cache=True).past_key_values
    ids = torch.zeros(1, 1, dtype=torch.long, device=DEV)
    return lambda: model(input_ids=ids, past_key_values=cache, use_cache=True)

rows, name_rows = [], []
for label, fn in (('prefill, 128 tokens', one_prefill(128)),
                  ('prefill, 512 tokens', one_prefill(512)),
                  ('decode, context 512', one_decode(512))):
    n, gpu_ms, names = profile_pass(fn)
    rows.append({'pass': label, 'cuda_kernels': n, 'template_entries': 242,
                 'extra_vs_template': n - 242, 'gpu_ms_per_pass': gpu_ms})
    name_rows += [{'pass': label, 'kernel': k, 'per_pass': c / ACTIVE} for k, c in names.items()]
KCOUNT = pd.DataFrame(rows)
KNAMES = pd.DataFrame(name_rows).sort_values(['pass', 'per_pass'], ascending=[True, False])
display(KCOUNT.round(3))

for label, g in KNAMES.groupby('pass', sort=False):
    print()
    print(f'{label}: kernels per pass, most frequent first')
    for _, r in g.head(15).iterrows():
        print(f'  {r.per_pass:6.1f}  {r.kernel[:110]}')

## 5 · The real runs

Our scheduler, untouched, on the real clock. Each step's pieces run as real forward passes, then `torch.cuda.synchronize()` makes the timing real. Every step records its start, its synchronized end, the exact pieces it ran and the scheduler's queue and KV state. Waits for the next arrival are recorded as idle.

**Energy from NVML.** On an A100 the energy counter only updates about every 100 ms. The helpers below keep just the moments it changed, so interpolating between them spreads each update over the time it actually covers. The last validation run interpolated between every 10 ms sample instead, which made 0.25 s windows flip between +14 W and -7 W.

In [ ]:
from dynshape import ChunkedPrefillScheduler, ModelConfig, HardwareConfig

def run_real(requests, sched_cfg, executor, sample_s=0.01, pad_s=1.0):
    '''Our scheduler on the real clock: one forward pass per piece, NVML alongside.'''
    sched = ChunkedPrefillScheduler(sched_cfg, ModelConfig(), HardwareConfig())
    pending = sorted(requests, key=lambda r: (r.arrived_at, r.id))
    for r in pending:
        executor.admit(r.id, r.num_prefill_tokens)

    sampler = NvmlSampler(sample_s).start()
    time.sleep(pad_s)                              # a second of idle before the first request
    t0 = time.perf_counter()
    work_start_s = t0 - sampler.t0                 # traffic time 0, on the sampler clock

    steps, idles = [], []
    cursor = done = 0
    truncated = False
    while done < len(pending):
        now = time.perf_counter() - t0
        if now > MAX_RUN_S:
            truncated = True
            break
        while cursor < len(pending) and pending[cursor].arrived_at <= now:
            sched.add_request(pending[cursor])
            cursor += 1

        batch = sched.get_next_batch()
        if batch is None:
            if cursor >= len(pending):
                raise RuntimeError('nothing runnable and nothing left to arrive')
            a = time.perf_counter() - t0
            time.sleep(max(0.0, pending[cursor].arrived_at - a))
            idles.append((a, time.perf_counter() - t0))
            continue

        pieces = batch.pieces
        t_it = time.perf_counter() - t0
        batch.on_schedule(t_it)
        for p in pieces:
            executor.run_piece(p)
        gpu_sync()
        t_end = time.perf_counter() - t0
        steps.append(dict(t_start_s=t_it, t_end_s=t_end,
                          pieces=[(p.kind, p.request_id, p.tokens, p.context) for p in pieces],
                          n_waiting=sched.num_waiting, n_running=sched.num_running,
                          kv_blocks=sched.allocator.num_allocated_blocks,
                          kv_utilisation=sched.allocator.utilisation,
                          preemptions_total=sched.num_preemptions))

        batch.on_batch_end(time.perf_counter() - t0)
        sched.on_batch_end(batch)
        for r in batch.requests:
            if r.completed:
                executor.release(r.id)
                done += 1

    work_end_s = time.perf_counter() - t0
    time.sleep(pad_s)                              # and a second after the last
    nv = sampler.stop()
    nv['t_work_s'] = nv.t_s - work_start_s         # sampler time on the traffic clock
    return dict(steps=steps, idles=idles, nvml=nv, work_end_s=work_end_s,
                truncated=truncated, requests=pending, sched_report=sched.report())


def cumulative_energy(nv):
    '''(t, E) on the traffic clock. With an energy counter, keep only the moments it
    changed; without one, integrate board power.'''
    t = nv.t_work_s.to_numpy()
    if HAVE_ENERGY:
        e = nv.energy_j.to_numpy()
        keep = np.r_[True, np.diff(e) != 0]
        return t[keep], e[keep]
    w = nv.power_w.to_numpy()
    return t, np.r_[0.0, np.cumsum(0.5 * (w[1:] + w[:-1]) * np.diff(t))]


def energy_between(cum, a, b):
    '''Joules between traffic times a and b.'''
    t, e = cum
    return float(np.interp(b, t, e) - np.interp(a, t, e))

print('run loop defined')

In [ ]:
REAL_RUNS = {}
for name in RUN:
    REAL_RUNS[name] = []
    for k in range(REPEATS):
        t0 = time.time()
        run = run_real(make_traffic(name), sched_for(name), Executor())
        REAL_RUNS[name].append(run)
        nv, wall = run['nvml'], run['work_end_s']
        during = nv[(nv.t_work_s >= 0) & (nv.t_work_s <= wall)]
        status = 'TRUNCATED at MAX_RUN_S' if run['truncated'] else 'complete'
        n_steps = len(run['steps'])
        print(f'{name:7s} run {k + 1}: {n_steps:5d} steps in {wall:6.1f} s, '
              f'clock {during.sm_mhz.median():.0f} MHz (min {during.sm_mhz.min():.0f}), '
              f'{status}   [{time.time() - t0:.0f} s]')

## 5b · CPU time per step, fitted from the real runs

HuggingFace spends about 16 ms of CPU on every forward pass while the GPU mostly waits, so a real step lasts as long as the CPU takes. Every real step above was timed, so a straight-line fit gives that cost:

**step time ≈ ms per pass × passes + ms per token × tokens + ms per step**

With `HOST_TIME = True` the simulator uses it: a simulated step lasts the longer of its GPU kernel time and this CPU time, and the GPU waits out the difference at idle + `HOST_WAIT_W`. The gap between back-to-back steps (the scheduler's own time) is taken from the same runs. The constants are fitted on these runs, so the red line's timing is calibrated, not predicted: the energy comparison is unaffected.

In [ ]:
rows, gaps = [], []
for name in RUN:
    for run in REAL_RUNS[name]:
        st = run['steps']
        for i, s in enumerate(st):
            rows.append(dict(condition=name, step_ms=1000.0 * (s['t_end_s'] - s['t_start_s']),
                             passes=len(s['pieces']), tokens=sum(p[2] for p in s['pieces'])))
            if i + 1 < len(st):
                g = 1000.0 * (st[i + 1]['t_start_s'] - s['t_end_s'])
                if 0.0 <= g < 20.0:                      # back to back, no idle wait between
                    gaps.append(g)
HOSTDATA = pd.DataFrame(rows)
A = np.c_[HOSTDATA['passes'], HOSTDATA['tokens'], np.ones(len(HOSTDATA))]
coef, *_ = np.linalg.lstsq(A, HOSTDATA['step_ms'].to_numpy(), rcond=None)
per_pass, per_token, per_step = (max(0.0, float(c)) for c in coef)
fit = A @ np.array([per_pass, per_token, per_step])
resid = HOSTDATA['step_ms'].to_numpy() - fit
r2 = 1.0 - float((resid ** 2).sum() / ((HOSTDATA['step_ms'] - HOSTDATA['step_ms'].mean()) ** 2).sum())
GAP_MS_REAL = float(np.median(gaps)) if gaps else 0.05
HOST = (dict(host_ms_per_pass=per_pass, host_ms_per_token=per_token, host_ms_per_step=per_step,
             host_wait_w=HOST_WAIT_W, gap_ms=GAP_MS_REAL) if HOST_TIME else {})
print(f'real step time = {per_pass:.2f} ms per pass + {per_token:.4f} ms per token '
      f'+ {per_step:.2f} ms per step   (R^2 {r2:.3f} over {len(HOSTDATA)} steps)')
print(f'gap between back-to-back steps: {GAP_MS_REAL:.2f} ms (median of {len(gaps)})')
print('the simulator will use these' if HOST_TIME else 'HOST_TIME is off: kernel time only')
display(HOSTDATA.assign(fit_ms=fit).groupby('condition')[['step_ms', 'fit_ms']].median().round(2))

## 5c · The idle floor during each run

The last run showed the card idling 1-2 W **above** the idle measured once at the start, in the gaps between bursts, while the simulator drops to exactly idle there. So each run's floor is measured from its own idle gaps: the waits for the next request, each trimmed by 0.3 s so no step energy leaks in, used when they add up to at least 2 s. A run with fewer idle gaps (`bursty`, `prefill_bursty`) keeps the start floor.

The real line and the real energy above idle use this floor from here on. The summary also keeps the comparison against the start floor (`*_start_floor` columns), so both are visible.

In [ ]:
def in_run_idle_w(run, trim_s=0.3, min_gap_s=0.5, need_s=2.0):
    '''(watts, seconds used, source): board power while the card waited for the next request
    during this run, from its idle gaps; the start floor if they add up to under need_s.'''
    cum = cumulative_energy(run['nvml'])
    spans = [(a + trim_s, b) for a, b in run['idles'] if b - a >= min_gap_s + trim_s]
    secs = sum(b - a for a, b in spans)
    if secs < need_s:
        return IDLE_W, secs, 'start (too few idle gaps)'
    return sum(energy_between(cum, a, b) for a, b in spans) / secs, secs, 'idle gaps'

FLOOR, rows = {}, []
for name in RUN:
    FLOOR[name] = []
    for k, run in enumerate(REAL_RUNS[name]):
        w, secs, src = in_run_idle_w(run)
        FLOOR[name].append(w)
        rows.append(dict(condition=name, run=k + 1, start_idle_w=IDLE_W, in_run_idle_w=w,
                         above_start_w=w - IDLE_W, idle_gap_s=secs, floor_from=src))
FLOORS = pd.DataFrame(rows)
display(FLOORS.round(2))

## 6 · The simulator: EnergAIzer, with bucketing

The same predictor as `EnergAIzer_LUT_Fast_Colab.ipynb`: the full per-shape fit, with decode contexts rounded to multiples of `BUCKET`. Building it is the slow one-time step.

With `EXTRAS = True` every forward pass also gets the kernels the template leaves out: 338 kernels per prefill pass and 326 per decode step instead of 242, the counts profiled on this executor. Bucketing applies to them too. `A100_Extra_Kernels_Colab.ipynb` checks the extras pass by pass.

In [ ]:
from dynshape import ShapeRewriter
from dynshape.energaizer import (clone_artifact, download_lut, lut_status, locate_artifact,
                                 build_gee_predictor, measured_idle_power_w)

ART = clone_artifact('/content/energaizer')
download_lut(ART)
st = lut_status(locate_artifact(ART))
assert st['complete'], 'LUT incomplete: ' + str(st['missing'])

t0 = time.time()
rw  = ShapeRewriter.from_dir('templates/gpt2')
gee = build_gee_predictor(ART)
LUT_IDLE_W = measured_idle_power_w(ART, 900)      # the floor inside every predicted kernel
print(f'built in {time.time() - t0:.0f} s; EnergAIzer idle floor {LUT_IDLE_W:.2f} W')

In [ ]:
from contextlib import contextmanager
import dynshape.mixed as mixed
from dynshape import Piece
from dynshape.mixed import build_iteration_kernels_tagged

# The original, exact expanders -- kept so rerunning this cell never wraps a wrapper.
# `_piece_extras` sizes the extra kernels; it is None on a checkout without them.
_exact_piece_entries = getattr(mixed, '_exact_piece_entries', mixed._piece_entries)
mixed._exact_piece_entries = _exact_piece_entries
_exact_piece_extras = getattr(mixed, '_exact_piece_extras', getattr(mixed, '_piece_extras', None))
mixed._exact_piece_extras = _exact_piece_extras

def bucket_context(ctx, size=None):
    '''Round a decode context to the nearest multiple of size, never below size.'''
    size = BUCKET if size is None else size
    return max(size, int(round(ctx / size)) * size)

def _bucketed(piece):
    if piece.kind == 'decode' and BUCKET > 1:
        return dataclasses.replace(piece, context=bucket_context(piece.context))
    return piece

def _bucketed_piece_entries(rewriter, piece, indices=None):
    return _exact_piece_entries(rewriter, _bucketed(piece), indices)

def _bucketed_piece_extras(extras, piece):
    return _exact_piece_extras(extras, _bucketed(piece))

def set_bucketing(on):
    '''Bucket decode contexts for the template and for the extra kernels, or for neither.'''
    mixed._piece_entries = _bucketed_piece_entries if on else _exact_piece_entries
    if _exact_piece_extras is not None:
        mixed._piece_extras = _bucketed_piece_extras if on else _exact_piece_extras

set_bucketing(BUCKET > 1)

@contextmanager
def exact_contexts():
    '''Temporarily switch bucketing off -- for the shapes the GPU really ran.'''
    saved = (mixed._piece_entries, getattr(mixed, '_piece_extras', None))
    set_bucketing(False)
    try:
        yield
    finally:
        mixed._piece_entries = saved[0]
        if saved[1] is not None:
            mixed._piece_extras = saved[1]

print(f'BUCKET = {BUCKET}' + ('  (exact contexts)' if BUCKET <= 1 else ''))

## 6b · Measured prices for the LM head and the mask kernels

The extra-kernel check (`A100_Extra_Kernels_Colab.ipynb`) found three kernels EnergAIzer prices badly:

| kernel | real vs EnergAIzer's stand-in | why |
|---|---|---|
| **LM head** (tokens × 50,257 × 768 GEMM) | 3.4-3.8× slower for prefills | GPT-2's vocabulary is odd, so cuBLAS falls back to slow unaligned kernels |
| **causal `where`** (12 heads × query × key) | 1.6× slower at 1,000 tokens | PyTorch broadcasts the mask across heads in its unvectorised kernel |
| **attention-mask add** (same shape) | same broadcast | same reason |

So they are timed here, **on this card**, and priced from the measurement:

- **time**: the profiled GPU time of one call
- **energy**: NVML's energy counter over 2.5 s of back-to-back calls, minus this card's idle, per call

The LM head is measured at 15 token counts (1 to 4,096); the mask kernels at 10 (query, key) shapes, from one decode token to a full 1,024-token prefill. Between sizes the price is interpolated. About 3 minutes. `MEASURE_PRICES = False` skips this and keeps EnergAIzer's stand-ins.

The first table puts each measurement next to the stand-in EnergAIzer used (`time_ratio`, `energy_ratio`: measured / EnergAIzer). The second shows what the new prices do to single passes, against the GPU work per pass measured in `A100_Extra_Kernels_Colab.ipynb`.

In [ ]:
from torch.profiler import profile, ProfilerActivity
from dynshape.engine import price_kernel
from dynshape.extras import MeasuredPrice, gpt2_hf_eager

LM_SIZES    = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 768, 1000, 1024, 2048, 4096]
MASK_SHAPES = [(1, 64), (1, 256), (1, 1024), (64, 64), (128, 128), (256, 256),
               (512, 512), (768, 768), (1000, 1000), (1024, 1024)]          # (query, key)
BENCH_S     = 2.5          # seconds of back-to-back calls per size, for the energy counter


def _cum(nv):
    '''(t, E) on the sampler's own clock, keeping only the moments the counter changed.'''
    t = nv.t_s.to_numpy()
    if HAVE_ENERGY:
        e = nv.energy_j.to_numpy()
        keep = np.r_[True, np.diff(e) != 0]
        return t[keep], e[keep]
    w = nv.power_w.to_numpy()
    return t, np.r_[0.0, np.cumsum(0.5 * (w[1:] + w[:-1]) * np.diff(t))]


def gpu_ms_per_call(fn, calls=10):
    '''(GPU ms, kernels) per call: profiled kernel durations, averaged over calls.'''
    for _ in range(3):
        fn()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for _ in range(calls):
            fn()
        torch.cuda.synchronize()
    prof.export_chrome_trace('/content/bench_trace.json')
    ker = [e for e in json.load(open('/content/bench_trace.json'))['traceEvents']
           if e.get('cat') in ('kernel', 'gpu_memcpy', 'gpu_memset')]
    return sum(e['dur'] for e in ker) / calls / 1000.0, len(ker) / calls


def graphed(fn, n=50):
    '''n calls of fn captured in one CUDA graph, so tiny kernels run back to back.'''
    try:
        side = torch.cuda.Stream()
        side.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(side):
            for _ in range(3):
                fn()
        torch.cuda.current_stream().wait_stream(side)
        graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(graph):
            for _ in range(n):
                fn()
        torch.cuda.synchronize()
        return graph.replay, n
    except Exception as exc:
        torch.cuda.synchronize()
        print(f'    (graph capture failed: {type(exc).__name__}; using plain calls)')
        return fn, 1


def dyn_mj_per_call(fn, gpu_ms):
    '''(mJ above this card's idle, wall ms) per call, over BENCH_S of back-to-back calls.'''
    run, per = graphed(fn) if gpu_ms < 0.05 else (fn, 1)
    gpu_per_run = max(gpu_ms * per, 1e-3)
    for _ in range(max(3, min(2000, int(300.0 / gpu_per_run)))):     # ~0.3 s of warm-up
        run()
    torch.cuda.synchronize()
    sync_every = max(1, int(100.0 / gpu_per_run))                    # at most ~0.1 s queued
    smp = NvmlSampler(0.01).start()
    time.sleep(0.3)
    start, runs = time.perf_counter(), 0
    while time.perf_counter() - start < BENCH_S:
        run()
        runs += 1
        if runs % sync_every == 0:
            torch.cuda.synchronize()
    torch.cuda.synchronize()
    stop = time.perf_counter()
    time.sleep(0.3)
    nv = smp.stop()
    a, b = start - smp.t0, stop - smp.t0
    t, e = _cum(nv)
    joules = float(np.interp(b, t, e) - np.interp(a, t, e))
    calls = runs * per
    return 1000.0 * (joules - IDLE_W * (b - a)) / calls, 1000.0 * (b - a) / calls


X0 = gpt2_hf_eager()                     # EnergAIzer's stand-ins, for comparison

def energaizer_price(name, q, k):
    '''(ms, mJ above EnergAIzer's idle floor) of the stand-in the simulator used before.'''
    eq, eop = X0.kernel(name).entry(q, k)
    t_ms, _p, e_j = gee.predict(eq, eop)
    return t_ms, 1000.0 * e_j - LUT_IDLE_W * t_ms


BENCH = []

def bench(name, size, q, k, fn):
    ms, nk = gpu_ms_per_call(fn)
    dyn, wall = dyn_mj_per_call(fn, ms)
    if ms <= 0:                     # the profiler saw no kernel: use wall time per call
        ms = wall
    e_ms, e_mj = energaizer_price(name, q, k)
    BENCH.append(dict(kernel=name, size=size, q=q, k=k, gpu_ms=ms, kernels_per_call=nk,
                      dyn_mj=dyn, power_above_idle_w=dyn / ms, wall_ms_per_call=wall,
                      energaizer_ms=e_ms, energaizer_dyn_mj=e_mj))
    print(f'  {name:14s} {size:>12,}  {ms:8.4f} ms {dyn:9.4f} mJ  | '
          f'EnergAIzer {e_ms:8.4f} ms {e_mj:9.4f} mJ')


if MEASURE_PRICES and XS is not None:
    t0 = time.time()
    lm = model.lm_head
    print('LM head: tokens x 50257 x 768, as the executor calls it')
    for m in LM_SIZES:
        x = torch.randn(1, m, 768, device=DEV, dtype=torch.bfloat16)
        bench('lm_head', m, m, m, lambda: lm(x))
        del x
    tril = torch.tril(torch.ones(1024, 1024, dtype=torch.bool, device=DEV)).view(1, 1, 1024, 1024)
    minv = torch.full([], torch.finfo(torch.bfloat16).min, dtype=torch.bfloat16, device=DEV)
    print('mask kernels: 12 heads x query x key, as eager GPT-2 runs them')
    for q, k in MASK_SHAPES:
        scores = torch.randn(1, 12, q, k, device=DEV, dtype=torch.bfloat16)
        amask = torch.zeros(1, 1, q, k, device=DEV, dtype=torch.bfloat16)
        cmask = tril[:, :, k - q:k, :k]           # the causal buffer's slice, as in GPT2Attention
        bench('causal_where', 12 * q * k, q, k, lambda: torch.where(cmask, scores, minv))
        bench('attn_mask_add', 12 * q * k, q, k, lambda: scores + amask)
        del scores, amask
    torch.cuda.empty_cache()
    BENCH = pd.DataFrame(BENCH)
    BENCH['time_ratio'] = BENCH.gpu_ms / BENCH.energaizer_ms
    BENCH['energy_ratio'] = BENCH.dyn_mj / BENCH.energaizer_dyn_mj
    display(BENCH.round(4))

    def measured(name, field):
        d = BENCH[BENCH.kernel == name].sort_values('size')
        return MeasuredPrice(field=field, sizes=tuple(d['size']), time_ms=tuple(d['gpu_ms']),
                             dyn_mj=tuple(d['dyn_mj']),
                             source=f'{CARD} at {TARGET_MHZ} MHz, A100_Validation_Colab.ipynb')

    XS = XS.with_prices({'lm_head': measured('lm_head', 'dimM'),
                         'causal_where': measured('causal_where', 'dim'),
                         'attn_mask_add': measured('attn_mask_add', 'dim')})
    print(f"priced from these measurements: {', '.join(XS.priced)}   [{time.time() - t0:.0f} s]")
else:
    BENCH = pd.DataFrame()
    print('MEASURE_PRICES is off (or EXTRAS is): EnergAIzer prices every kernel')

In [ ]:
# Single passes, priced three ways, against the GPU work per pass (energy above idle,
# CPU waiting taken out) measured in A100_Extra_Kernels_Colab.ipynb on an A100-SXM4.
REFERENCE = [('prefill 1000', 'prefill', 1000, 0, 948.6), ('prefill 512', 'prefill', 512, 0, 312.6),
             ('prefill 128', 'prefill', 128, 0, 91.8), ('decode @128', 'decode', 1, 128, 48.6),
             ('decode @512', 'decode', 1, 512, 56.2), ('decode @1000', 'decode', 1, 1000, 66.9)]


def pass_dyn_mj(piece, extras):
    '''Simulated energy above idle for one forward pass, in mJ.'''
    t_ms = e_j = 0.0
    for (q, op), tag in build_iteration_kernels_tagged(rw, [piece], fuse_linear=False,
                                                       extras=extras):
        kt, _kp, ke = price_kernel(gee, q, op, tag, extras, LUT_IDLE_W)
        t_ms += kt
        e_j += ke
    return 1000.0 * e_j - LUT_IDLE_W * t_ms


if XS is None:
    PASSCHECK = pd.DataFrame()
    print('EXTRAS is off: nothing to compare')
else:
    rows = []
    with exact_contexts():
        for label, kind, tokens, ctx, ref in REFERENCE:
            piece = Piece(kind=kind, request_id=0, tokens=tokens, context=ctx)
            rows.append(dict(case=label, measured_gpu_work_mj=ref,
                             template_mj=pass_dyn_mj(piece, None),
                             extras_energaizer_mj=pass_dyn_mj(piece, X0),
                             extras_measured_mj=pass_dyn_mj(piece, XS)))
    PASSCHECK = pd.DataFrame(rows).set_index('case')
    for col in ('template_mj', 'extras_energaizer_mj', 'extras_measured_mj'):
        PASSCHECK[col.replace('_mj', '_err_pct')] = \
            100.0 * (PASSCHECK[col] / PASSCHECK.measured_gpu_work_mj - 1.0)
    display(PASSCHECK.round(1))
    for label, r in PASSCHECK.iterrows():
        print(f'{label:13s} measured {r.measured_gpu_work_mj:6.1f} mJ | template '
              f'{r.template_err_pct:+5.0f}% | + extras (EnergAIzer) {r.extras_energaizer_err_pct:+5.0f}% '
              f'| + extras (measured prices) {r.extras_measured_err_pct:+5.0f}%')

## 7 · Simulate each condition, and price the real steps

For each condition this builds three traces, all ordinary `EngineTrace` objects, so they share one CSV format:

- **sim**: `run_engine` on the same traffic, its own clock.
- **real**: one row per real step. Times are measured, energy comes from NVML, and batch, token, KV and FLOP columns are computed from the pieces that ran, exactly the way the simulator computes them.
- **sim, same steps**: the real steps again, with energy = this card's idle floor + the simulator's predicted dynamic energy for those exact pieces (EnergAIzer, and the measured prices of section 6b where a kernel has one).

**Energy above idle** is what gets compared. Each side subtracts its own floor (this card's measured idle for the real run, EnergAIzer's 47.35 W for its kernels), so the SXM-vs-PCIe idle difference drops out. Board power is in the summary too, but it is mostly idle floor on a model this small.

In [ ]:
from dynshape import (EngineConfig, EngineTrace, IterationRecord, Segment, run_engine,
                      kernel_work, WORK_FIELDS, iteration_token_shapes)
from dynshape.engine import price_kernel

OUT = 'compare_out'
os.makedirs(OUT, exist_ok=True)
SKIPPED = collections.Counter()

def expand(pieces, exact):
    '''Recorded pieces -> (Piece objects, tagged kernels), one forward pass per piece.'''
    ps = [Piece(kind=k, request_id=r, tokens=t, context=c) for k, r, t, c in pieces]
    with (exact_contexts() if exact else contextlib.nullcontext()):
        return ps, build_iteration_kernels_tagged(rw, ps, fuse_linear=False, extras=XS)

def work_vector(ps, tagged):
    '''Batch columns and FLOPs/bytes -- the same accounting execute_iteration uses.'''
    sh = iteration_token_shapes(ps)
    share = sh['prefill_tokens'] / sh['total_tokens'] if sh['total_tokens'] else 0.0
    w = {f: 0.0 for f in WORK_FIELDS}
    for (q, op), tag in tagged:
        phase = tag.partition('+')[0]          # an extra kernel is tagged '<phase>+<name>'
        fl, by, wb = kernel_work(q, op, strict=False)
        if phase.endswith('prefill') or phase.endswith('decode'):
            w['attn_flops'] += fl
            w['attn_bytes'] += by
            w['prefill_flops' if phase.endswith('prefill') else 'decode_flops'] += fl
        else:
            w['linear_flops'] += fl
            w['linear_bytes'] += by
            w['weight_bytes'] += wb
            w['prefill_flops'] += fl * share
            w['decode_flops'] += fl * (1.0 - share)
    return sh, w

def predicted_dynamic(tagged):
    '''(dynamic J, kernel ms): energy above the idle floor, priced exactly as run_engine
    prices it -- measured prices for the extras that have one, EnergAIzer for the rest.'''
    t_ms = e_j = 0.0
    for (q, op), tag in tagged:
        try:
            kt, _kp, ke = price_kernel(gee, q, op, tag, XS, LUT_IDLE_W)
        except Exception:
            SKIPPED[op[0]] += 1
            continue
        t_ms += kt
        e_j += ke
    return e_j - LUT_IDLE_W * t_ms / 1000.0, t_ms


def build_traces(name, run):
    '''Two EngineTraces on the REAL timeline -- what NVML measured, and what the simulator
    predicts for the very same steps -- plus one row per step for the step-time plot.'''
    nv, end = run['nvml'], run['work_end_s']
    cum = cumulative_energy(nv)
    events = sorted([(s['t_start_s'], 'step', s) for s in run['steps']]
                    + [(a, 'idle', None) for a, _b in run['idles']], key=lambda ev: ev[0])
    starts = [ev[0] for ev in events] + [end]
    cfg = EngineConfig(scheduler=sched_for(name), fuse_linear=False, idle_w=IDLE_W, extras=XS)
    real = EngineTrace(requests=list(run['requests']), scheduler_report=run['sched_report'],
                       config=cfg)
    pred = EngineTrace(requests=list(run['requests']), scheduler_report=run['sched_report'],
                       config=cfg)
    rows = []

    def add_idle(a, b):
        e = energy_between(cum, a, b)
        real.idle_segments.append(Segment(kind='IDLE', t_start_ms=1000 * a, time_ms=1000 * (b - a),
                                          power_w=e / (b - a), energy_j=e, op='idle'))
        pred.idle_segments.append(Segment(kind='IDLE', t_start_ms=1000 * a, time_ms=1000 * (b - a),
                                          power_w=IDLE_W, energy_j=IDLE_W * (b - a), op='idle'))

    if starts[0] > 0:
        add_idle(0.0, starts[0])
    # Every moment belongs to exactly one event: a step (its GPU span, then the gap until
    # the next event) or an idle wait. So the energies add up to the whole run.
    for i, (a, kind, s) in enumerate(events):
        b = starts[i + 1]
        if b <= a:
            continue
        if kind == 'idle':
            add_idle(a, b)
            continue
        dur = s['t_end_s'] - a
        span = max(b - a, dur)
        gap = span - dur
        ps, exact_kernels = expand(s['pieces'], exact=True)
        sh, work = work_vector(ps, exact_kernels)
        _, sim_kernels = expand(s['pieces'], exact=False)
        dyn_j, kern_ms = predicted_dynamic(sim_kernels)
        e_real = energy_between(cum, a, a + span)
        e_pred = IDLE_W * span + dyn_j
        inside = nv.power_w[(nv.t_work_s >= a) & (nv.t_work_s <= a + span)]
        common = dict(t_start_ms=1000 * a, duration_ms=1000 * dur, gap_ms=1000 * gap,
                      decode_batch=sh['decode_batch'], prefill_chunks=sh['prefill_chunks'],
                      prefill_tokens=sh['prefill_tokens'], decode_tokens=sh['decode_batch'],
                      total_tokens=sh['total_tokens'], context_mean=sh['context_mean'],
                      context_max=sh['context_max'], n_kernels=len(exact_kernels), n_skipped=0,
                      energy_fused_j=np.nan, energy_attn_prefill_j=np.nan,
                      energy_attn_decode_j=np.nan,
                      n_waiting=s['n_waiting'], n_running=s['n_running'],
                      kv_blocks=s['kv_blocks'], kv_utilisation=s['kv_utilisation'],
                      preemptions_total=s['preemptions_total'], **work)
        real.iterations.append(IterationRecord(
            idx=len(real.iterations), energy_j=e_real, avg_power_w=e_real / span,
            peak_power_w=float(inside.max()) if len(inside) else e_real / span, **common))
        pred.iterations.append(IterationRecord(
            idx=len(pred.iterations), energy_j=e_pred, avg_power_w=e_pred / span,
            peak_power_w=np.nan, **common))
        rows.append(dict(real_ms=1000 * dur, sim_kernel_ms=kern_ms, passes=len(ps),
                         prefill_tokens=sh['prefill_tokens']))
    return real, pred, rows


def dyn_energy(tr, floor):
    '''Energy above an idle floor, over the whole trace.'''
    return tr.total_energy_j - floor * tr.total_time_ms / 1000.0


SIM, REAL_TR, PRED_TR, STEPS = {}, {}, {}, []
for name in RUN:
    t0, shapes0 = time.time(), len(gee.cache)
    SIM[name] = run_engine(make_traffic(name), rw, gee,
                           EngineConfig(scheduler=sched_for(name), fuse_linear=False,
                                        record_kernels_until_ms=0.0, idle_w=LUT_IDLE_W,
                                        extras=XS, **HOST))
    REAL_TR[name], PRED_TR[name] = [], []
    for k, run in enumerate(REAL_RUNS[name]):
        real, pred, rows = build_traces(name, run)
        REAL_TR[name].append(real)
        PRED_TR[name].append(pred)
        STEPS += [dict(condition=name, run=k + 1, **r) for r in rows]
    new_shapes = len(gee.cache) - shapes0
    print(f'{name:14s} simulated and priced in {time.time() - t0:.0f} s ({new_shapes:,} new shapes)')
STEPS = pd.DataFrame(STEPS)
skipped_msg = dict(SKIPPED) or 'none'
print(f'kernels EnergAIzer could not price: {skipped_msg}')
if XS is not None:
    print('simulator:', SIM[RUN[0]].shape_report)

rows = []
for name in RUN:
    sim = SIM[name]
    ss = sim.summary()
    sim_wall = sim.total_time_ms / 1000.0
    sim_dyn = dyn_energy(sim, LUT_IDLE_W)
    for k, (real, pred) in enumerate(zip(REAL_TR[name], PRED_TR[name])):
        sr = real.summary()
        wall = real.total_time_ms / 1000.0
        floor = FLOOR[name][k]
        real_dyn = dyn_energy(real, floor)              # above the idle floor measured in this run
        real_dyn_start = dyn_energy(real, IDLE_W)       # above the idle measured once at the start
        rows.append({
            'condition': name, 'run': k + 1, 'requests': sr['requests'],
            'idle_w_start': IDLE_W, 'idle_w_in_run': floor, 'dyn_j_real_start_floor': real_dyn_start,
            'steps_real': sr['iterations'], 'steps_sim': ss['iterations'],
            'wall_s_real': wall, 'wall_s_sim': sim_wall,
            'dyn_j_real': real_dyn, 'dyn_j_sim_same_steps': dyn_energy(pred, IDLE_W),
            'dyn_j_sim': sim_dyn,
            'avg_dyn_w_real': real_dyn / wall, 'avg_dyn_w_sim': sim_dyn / sim_wall,
            'board_w_real': sr['avg_power_w_wallclock'], 'board_w_sim': ss['avg_power_w_wallclock'],
            'tokens_per_s_real': sr['output_tokens_per_s'],
            'tokens_per_s_sim': ss['output_tokens_per_s'],
            'ttft_p50_s_real': sr['ttft_p50_s'], 'ttft_p50_s_sim': ss['ttft_p50_s'],
            'itl_p50_ms_real': sr['itl_p50_ms'], 'itl_p50_ms_sim': ss['itl_p50_ms']})
SUMMARY = pd.DataFrame(rows)
SUMMARY['wall_ratio_sim_over_real'] = SUMMARY.wall_s_sim / SUMMARY.wall_s_real
SUMMARY['dyn_err_same_steps_pct'] = 100 * (SUMMARY.dyn_j_sim_same_steps / SUMMARY.dyn_j_real - 1)
SUMMARY['dyn_err_sim_pct'] = 100 * (SUMMARY.dyn_j_sim / SUMMARY.dyn_j_real - 1)
SUMMARY['dyn_err_same_steps_start_floor_pct'] = 100 * (SUMMARY.dyn_j_sim_same_steps / SUMMARY.dyn_j_real_start_floor - 1)
SUMMARY['dyn_err_sim_start_floor_pct'] = 100 * (SUMMARY.dyn_j_sim / SUMMARY.dyn_j_real_start_floor - 1)
display(SUMMARY.set_index(['condition', 'run']).T.round(3))

## 8 · Plots

Each condition gets two panels on one time axis:

- **power above idle**: black is the real A100, yellow is the simulator on its own clock. If yellow ends early, the simulator is missing *time*, not necessarily energy.
- **traffic** underneath: the dotted line counts requests arriving in each window; the bars count forward passes run in that window on the real GPU, prefill in purple and decode in green.

The real line is power above the idle floor measured during its own run (section 5c), the simulator above its own floor. Board power is this plus about 48-51 W. The simulator-on-the-real-steps view is in the summary table (`*_same_steps` columns), no longer plotted.

In [ ]:
SIM_COLOR = '#FFB300'                      # bright amber-yellow: the simulator, own clock
PREFILL_C, DECODE_C = '#6a3d9a', '#1b9e77'  # traffic bars: prefill purple, decode green


def dt_for(name):
    '''The plot and comparison window of a condition.'''
    return CONDITIONS[name].get('dt', PLOT_DT_S)


def dyn_series(tr, floor, total_s, dt_s):
    '''Power above an idle floor on a fixed grid, padded at the floor past the trace end.'''
    t_ms, p = tr.resample(dt_ms=1000 * dt_s, total_ms=1000 * total_s, pad_power_w=floor)
    return t_ms / 1000.0, p - floor


def traffic_series(run, total_s, dt_s):
    '''Per window: requests arriving, and the prefill and decode passes the real GPU ran.'''
    n = int(np.ceil(total_s / dt_s)) + 1
    arrivals, prefill, decode = np.zeros(n), np.zeros(n), np.zeros(n)
    for r in run['requests']:
        i = int(r.arrived_at // dt_s)
        if 0 <= i < n:
            arrivals[i] += 1
    for s in run['steps']:
        i = int(s['t_start_s'] // dt_s)
        if 0 <= i < n:
            for kind, _rid, _tokens, _ctx in s['pieces']:
                if kind == 'prefill':
                    prefill[i] += 1
                else:
                    decode[i] += 1
    return np.arange(n) * dt_s, arrivals, prefill, decode


SERIES, CORR = {}, {}
GROUPS = {}
for name in RUN:
    GROUPS.setdefault(CONDITIONS[name].get('scale', 'base'), []).append(name)

for tag, names in GROUPS.items():              # one figure per length: short, long
    fig = plt.figure(figsize=(13, 4.6 * len(names)))
    gs = fig.add_gridspec(2 * len(names), 1, height_ratios=[3.0, 1.3] * len(names), hspace=0.6)
    for j, name in enumerate(names):
        dt = dt_for(name)
        ax = fig.add_subplot(gs[2 * j])
        axt = fig.add_subplot(gs[2 * j + 1], sharex=ax)
        real, pred, sim = REAL_TR[name][0], PRED_TR[name][0], SIM[name]
        total = max(real.total_time_ms, sim.total_time_ms) / 1000.0
        t, p_real = dyn_series(real, FLOOR[name][0], total, dt)    # above this run's own idle floor
        _, p_pred = dyn_series(pred, IDLE_W, total, dt)
        _, p_sim = dyn_series(sim, LUT_IDLE_W, total, dt)
        n = min(len(t), len(p_pred), len(p_sim))
        SERIES[name] = pd.DataFrame({'t_s': t[:n], 'real_dyn_w': p_real[:n],
                                     'sim_same_steps_dyn_w': p_pred[:n], 'sim_dyn_w': p_sim[:n]})
        ser = SERIES[name]
        m = (ser.t_s < real.total_time_ms / 1000.0).to_numpy()
        CORR[name] = (float(np.corrcoef(ser.real_dyn_w[m], ser.sim_dyn_w[m])[0, 1])
                      if m.sum() > 2 else np.nan)
        ax.step(ser.t_s, ser.sim_dyn_w, where='post', color=SIM_COLOR, lw=2.6,
                label='simulator, its own clock')
        ax.step(ser.t_s, ser.real_dyn_w, where='post', color='black', lw=1.3,
                label='real A100 (NVML)')
        s = SUMMARY[(SUMMARY.condition == name) & (SUMMARY.run == 1)].iloc[0]
        ax.set_title(f'{name} ({dt:g} s windows): real {s.wall_s_real:.1f} s vs simulated '
                     f'{s.wall_s_sim:.1f} s   |   energy above idle: real {s.dyn_j_real:.0f} J, '
                     f'simulator {s.dyn_j_sim:.0f} J ({s.dyn_err_sim_pct:+.0f}%)   |   '
                     f'r = {CORR[name]:.2f}', fontsize=10)
        ax.set_ylabel('power above idle (W)')
        ax.grid(alpha=.25)
        ax.legend(prop={'size': 13, 'weight': 'bold'}, loc='upper right')
        plt.setp(ax.get_xticklabels(), visible=False)

        tt, arrivals, prefill, decode = traffic_series(REAL_RUNS[name][0], total, dt)
        arrivals, prefill, decode = arrivals / dt, prefill / dt, decode / dt   # per second
        axt.bar(tt, prefill, width=dt, align='edge', color=PREFILL_C, label='prefill passes')
        axt.bar(tt, decode, width=dt, align='edge', bottom=prefill, color=DECODE_C,
                label='decode passes')
        axt.step(tt, arrivals, where='post', color='black', lw=1.0, ls=':', label='requests arriving')
        axt.set_ylabel('forward passes/s (bars),\nrequests arriving/s (dotted)', fontsize=8)
        axt.set_xlabel('time from first request (s)', fontsize=8)
        axt.grid(alpha=.25)
        axt.legend(fontsize=7, loc='upper right', ncol=3)
    fig.savefig(f'{OUT}/fig_power_traces' + ('' if tag == 'base' else f'_{tag}') + '.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
LABEL = {'real': 'real A100', 'sim': 'simulator, own clock'}
COLOR = {'real': 'black', 'sim': SIM_COLOR}
S1 = SUMMARY[SUMMARY.run == 1].set_index('condition').loc[RUN]
panels = [('time the run took (s)', [('real', 'wall_s_real'), ('sim', 'wall_s_sim')], False),
          ('energy above idle (J, log scale)', [('real', 'dyn_j_real'), ('sim', 'dyn_j_sim')], True),
          ('average power above idle (W)', [('real', 'avg_dyn_w_real'), ('sim', 'avg_dyn_w_sim')], False)]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
x = np.arange(len(RUN))
for ax, (title, bars, logy) in zip(axes, panels):
    w = 0.8 / len(bars)
    for j, (who, col) in enumerate(bars):
        ax.bar(x + (j - (len(bars) - 1) / 2) * w, S1[col].to_numpy(), w,
               color=COLOR[who], edgecolor='#555555', linewidth=0.4, label=LABEL[who])
    if logy:
        ax.set_yscale('log')
    ax.set_xticks(x)
    ax.set_xticklabels(RUN, rotation=30, ha='right', fontsize=8)
    ax.set_title(title)
    ax.grid(alpha=.25, axis='y')
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(f'{OUT}/fig_summary_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
palette = ['#2ca02c', '#1f77b4', '#ff7f0e', '#9467bd', '#d62728', '#8c564b',
           '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
for name, c in zip(RUN, palette * 3):
    g = STEPS[(STEPS.condition == name) & (STEPS.run == 1)]
    ax[0].scatter(g.sim_kernel_ms, g.real_ms, s=7, alpha=.5, color=c, label=name)
    ax[1].scatter(g.passes, g.real_ms - g.sim_kernel_ms, s=7, alpha=.5, color=c, label=name)
lim = [0, max(STEPS.real_ms.max(), STEPS.sim_kernel_ms.max())]
ax[0].plot(lim, lim, 'k--', lw=.8, label='equal')
ax[0].set_xlabel('simulator: kernel time for the step (ms)')
ax[0].set_ylabel('real: measured step time (ms)')
ax[0].set_title('step time, same steps')
ax[1].set_xlabel('forward passes in the step')
ax[1].set_ylabel('real minus simulated (ms)')
ax[1].set_title('time the simulator does not model')
for a in ax:
    a.grid(alpha=.25)
    a.legend(fontsize=8)
plt.tight_layout()
fig.savefig(f'{OUT}/fig_step_time.png', dpi=150, bbox_inches='tight')
plt.show()

per_pass = (STEPS.real_ms - STEPS.sim_kernel_ms) / STEPS.passes
OVERHEAD = STEPS.assign(per_pass=per_pass).groupby('condition').per_pass.median().loc[RUN]
for name, v in OVERHEAD.items():
    print(f'{name:14s} real minus simulated time per forward pass: {v:.2f} ms (median)')

## 9 · Export

| file | what it is |
|---|---|
| `real_CONDITION_iterations.csv` | one row per real step, same 34 columns as `Bucketing_measured_iterations.csv` |
| `sim_CONDITION_iterations.csv` | one row per simulated step, same 34 columns |
| `sim_same_steps_CONDITION_iterations.csv` | the real steps priced by the simulator, same 34 columns |
| `real_/sim_CONDITION_requests.csv` | per request: arrival, TTFT, inter-token latency |
| `power_1000ms_CONDITION.csv` | the plotted series: power above idle, three lines |
| `nvml_CONDITION.csv` | raw NVML samples (power, energy counter, clock) |
| `summary.csv`, `steps_real_vs_sim.csv`, `kcount.csv`, `knames.csv` | the tables above, and the kernel names per pass |
| `fig_*.png` | the three figures |

With `REPEATS` above 1, real files get a `_runN` suffix.

In [ ]:
import zipfile

def sfx(k):
    return '' if REPEATS == 1 else f'_run{k + 1}'

for name in RUN:
    sim_iters, sim_reqs, _ = SIM[name].to_dataframes()
    sim_iters.to_csv(f'{OUT}/sim_{name}_iterations.csv', index=False)
    sim_reqs.to_csv(f'{OUT}/sim_{name}_requests.csv', index=False)
    for k, (real, pred) in enumerate(zip(REAL_TR[name], PRED_TR[name])):
        real_iters, real_reqs, _ = real.to_dataframes()
        real_iters.to_csv(f'{OUT}/real_{name}{sfx(k)}_iterations.csv', index=False)
        real_reqs.to_csv(f'{OUT}/real_{name}{sfx(k)}_requests.csv', index=False)
        pred.to_dataframes()[0].to_csv(f'{OUT}/sim_same_steps_{name}{sfx(k)}_iterations.csv',
                                       index=False)
        REAL_RUNS[name][k]['nvml'].to_csv(f'{OUT}/nvml_{name}{sfx(k)}.csv', index=False)
    SERIES[name].to_csv(f'{OUT}/power_{int(dt_for(name) * 1000)}ms_{name}.csv', index=False)

SUMMARY.to_csv(f'{OUT}/summary.csv', index=False)
STEPS.to_csv(f'{OUT}/steps_real_vs_sim.csv', index=False)
TRAFFIC.to_csv(f'{OUT}/traffic.csv')
for tname in ('KCOUNT', 'KNAMES'):
    if tname in globals():
        globals()[tname].to_csv(f'{OUT}/{tname.lower()}.csv', index=False)
if XS is not None:
    XS.save(f'{OUT}/extras.json')                       # with the measured prices, if any
if len(BENCH):
    BENCH.to_csv(f'{OUT}/measured_prices.csv', index=False)
if len(PASSCHECK):
    PASSCHECK.to_csv(f'{OUT}/pass_check.csv')
FLOORS.to_csv(f'{OUT}/idle_floors.csv', index=False)
pd.DataFrame([{'conditions': ','.join(RUN), 'repeats': REPEATS, 'bucket': BUCKET,
               'extras': XS.name if XS is not None else 'off',
               'measured_prices': ','.join(XS.priced) if XS is not None else '',
               'branch': BRANCH, 'target_mhz': TARGET_MHZ, 'card': CARD, 'idle_w': IDLE_W,
               'lut_idle_w': LUT_IDLE_W, 'plot_dt_s': PLOT_DT_S}]).to_csv(
    f'{OUT}/settings.csv', index=False)

with zipfile.ZipFile('sim_vs_real.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, f), f)
print(f'sim_vs_real.zip: {len(os.listdir(OUT))} files')
for f in sorted(os.listdir(OUT)):
    print('  ', f)

try:
    from google.colab import files
    files.download('sim_vs_real.zip')
except Exception:
    print('(not on Colab)')

## Notes

- **What earlier runs found.** The simulator was low on energy above idle and short on time. The template lacked 84-96 small kernels per pass and the LM head, and EnergAIzer prices the LM head and the mask kernels 1.6-3.8× too fast. Time is short because HuggingFace spends about 15 ms of CPU per forward pass, which the simulator does not model.
- **This run.** `EXTRAS = True` adds the missing kernels, and `MEASURE_PRICES = True` prices the LM head and the two mask kernels from timings on this card (section 6b). Compare `dyn_err_same_steps_pct` with the earlier runs: about -21% on decode-heavy traffic and -37% on 1,000-token prefills.
- **Decode-only set.** A 1-token prompt and 32-64 output tokens: every pass is a single token, so these test the many small kernels, the KV-cache appends and the CPU time per pass, with almost no prefill.
- **vLLM is not tested.** The real side is HuggingFace eager with one forward pass per request; a real serving engine fuses the linear layers across requests and uses paged attention.
- **Per-kernel peaks are not tested.** NVML cannot see a single kernel.
- **One card, one clock, one model:** A100-SXM4 at 900 MHz, GPT-2.